# Train Time-As-Channels 3D CNN

This notebook intentionally keeps orchestration thin. Split preparation, fitting, reporting, plotting, and persistence are delegated to shared utilities in `src.ml`.


In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd
import torch

from src.ml import (
    LossWeightConfig,
    OptimizationConfig,
    display_experiment_summary,
    display_holdout_evaluation,
    fit_estimator_on_experiment,
    persist_experiment_artifacts,
    plot_holdout_branch_embedding_projections,
    plot_holdout_embedding_projection,
    plot_training_history,
    prepare_multitask_experiment_data,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
from src.ml import TimeChannel3DCNNClassifier, TimeChannel3DCNNConfig


In [ ]:
# User inputs

dataset_artifact_path = "moa_4class_c2_mca1_mtc32_t30_z5_y96_x96.pt"
experiment_output_dir = Path("artifacts/nb6_time_channel_3dcnn")
persist_artifacts = True

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0

model_config = TimeChannel3DCNNConfig(
    conv_channels=(8, 16, 24, 32),
    kernel_size_z=(1, 1, 1, 1),
    kernel_size_xy=(5, 3, 3, 3),
    stride_z=(1, 1, 1, 1),
    stride_xy=(1, 1, 1, 1),
    pool_kernel_z=(1, 1, 1, 1),
    pool_kernel_xy=(2, 2, 2, 2),
    pool_stride_z=(1, 1, 1, 1),
    pool_stride_xy=(2, 2, 2, 2),
    embedding_dim=24,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=20,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=4,
    early_stopping_min_delta=0.0,
    scheduler_patience=1,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    compound_weight=0.2,
    concentration_weight=0.2,
)


In [ ]:
dataset = torch.load(dataset_artifact_path, map_location="cpu")


In [ ]:
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)


In [ ]:
display_experiment_summary(experiment)


In [ ]:
model = TimeChannel3DCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)


In [ ]:
fit_estimator_on_experiment(model, experiment)


In [ ]:
plot_training_history(model, title="Time-As-Channels 3D CNN loss curves");


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)


In [ ]:
holdout_embedding_projection = plot_holdout_embedding_projection(
    model,
    experiment,
    title="Holdout embedding projection by action",
)


In [ ]:
run_config = {
    "dataset_artifact_path": dataset_artifact_path,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}


In [ ]:
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
    )
